In [3]:
import anthropic
import os
import json
from pathlib import Path
import grpc
import networkx as nx
from pyvis.network import Network
from sentence_transformers import SentenceTransformer
import lancedb
from senzing import SzEngine, SzError, SzEngineFlags
from senzing_grpc import SzAbstractFactoryGrpc
from IPython.display import display, HTML, IFrame

print("All imports successful")

All imports successful


## Connect to Senzing and Export Entities

Opens a gRPC connection to the Senzing engine and streams the complete entity report, requesting the raw JSON for each source record alongside the resolved entity data. This is the same export used in notebook 04 and provides the data for building both graph structures.

In [4]:
SENZING_HOST = os.getenv('SENZING_GRPC_HOST', 'senzing')
SENZING_PORT = os.getenv('SENZING_GRPC_PORT', '8261')

print(f"Connecting to Senzing at {SENZING_HOST}:{SENZING_PORT}")

# Create gRPC channel and engine
grpc_url = f"{SENZING_HOST}:{SENZING_PORT}"
grpc_channel = grpc.insecure_channel(grpc_url)
sz_abstract_factory = SzAbstractFactoryGrpc(grpc_channel)
sz_engine = sz_abstract_factory.create_engine()

print("Connected to Senzing successfully")

# Export all resolved entities
print("\nExporting entity data from Senzing with full details...")

entities = []
export_handle = sz_engine.export_json_entity_report(
    flags=SzEngineFlags.SZ_EXPORT_INCLUDE_ALL_ENTITIES |
          SzEngineFlags.SZ_ENTITY_INCLUDE_RECORD_JSON_DATA |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ALL_RELATIONS |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ENTITY_NAME |
          SzEngineFlags.SZ_ENTITY_INCLUDE_RELATED_MATCHING_INFO
)

count = 0
while True:
    try:
        entity_json = sz_engine.fetch_next(export_handle)
        if not entity_json:
            break
        entities.append(json.loads(entity_json))
        count += 1
        if count % 50 == 0:
            print(f"  Exported {count} entities...", end='\r')
    except StopIteration:
        break

sz_engine.close_export_report(export_handle)
print(f"\nExported {len(entities)} entities total")

with_rels = sum(1 for e in entities if e.get('RELATED_ENTITIES'))
print(f"Entities with relationships: {with_rels}")

Connecting to Senzing at senzing:8261
Connected to Senzing successfully

Exporting entity data from Senzing with full details...
  Exported 150 entities...
Exported 196 entities total
Entities with relationships: 189


## Build True Combined Graph (Entities + Records + Relationships)

Constructs the two-layer graph from notebook 04: entity nodes (large shapes) represent resolved entities, record nodes (small dots) represent original source records. Gray dashed edges show which records were merged into each entity, red solid edges show business relationships between entities.

In [5]:
G_true_combined = nx.Graph()

entity_info = {}
entities_added = 0
records_added = 0

for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    records = entity_data.get('RECORDS', [])
    
    if not records:
        continue
    
    first_record = records[0]
    json_data = first_record.get('JSON_DATA', {})
    
    # v4 format: extract type and name from FEATURES
    record_type = 'UNKNOWN'
    name = None
    features = json_data.get('FEATURES', [])
    for feat in features:
        if 'RECORD_TYPE' in feat:
            record_type = feat['RECORD_TYPE']
        if not name:
            name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
    
    # Fallback to top-level fields
    if not name:
        name = json_data.get('PRIMARY_NAME_FULL')
    if not name:
        for name_obj in json_data.get('NAMES', []):
            name = name_obj.get('NAME_FULL') or name_obj.get('NAME_ORG') or name_obj.get('PRIMARY_NAME_ORG')
            if name:
                break
    if not name:
        name = f"Entity {entity_id}"
    
    data_sources = list(set([r.get('DATA_SOURCE') for r in records]))
    is_cross_source = len(data_sources) > 1
    
    entity_info[entity_id] = {
        'name': name,
        'type': record_type,
        'num_records': len(records),
        'is_cross_source': is_cross_source,
        'data_sources': data_sources
    }
    
    tooltip_parts = [
        name,
        f"Type: {record_type}",
        f"Entity ID: {entity_id}",
        f"Records merged: {len(records)}",
        f"Data sources: {', '.join(data_sources)}"
    ]
    if is_cross_source:
        tooltip_parts.append("CROSS-SOURCE RESOLUTION")
    tooltip = "\n".join(tooltip_parts)
    
    display_label = name[:30] + "..." if len(name) > 30 else name
    
    entity_node_id = f"entity_{entity_id}"
    G_true_combined.add_node(
        entity_node_id,
        label=display_label,
        title=tooltip,
        node_type='entity',
        type=record_type,
        num_records=len(records),
        is_cross_source=is_cross_source,
        entity_id=entity_id
    )
    entities_added += 1
    
    # Add record nodes and connect to entity
    for rec in records:
        rec_id = rec.get('RECORD_ID')
        rec_source = rec.get('DATA_SOURCE')
        rec_json = rec.get('JSON_DATA', {})
        
        # Get record name from FEATURES (v4)
        rec_name = None
        rec_type = 'UNKNOWN'
        for feat in rec_json.get('FEATURES', []):
            if 'RECORD_TYPE' in feat:
                rec_type = feat['RECORD_TYPE']
            if not rec_name:
                rec_name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
        if not rec_name:
            rec_name = name
        
        rec_tooltip = "\n".join([
            f"Record: {rec_name}",
            f"Source: {rec_source}",
            f"Type: {rec_type}",
            f"Record ID: {rec_id}",
            f"Resolves to: {name}"
        ])
        
        rec_label = rec_name[:20] + "..." if len(rec_name) > 20 else rec_name
        
        record_node_id = f"record_{rec_source}_{rec_id}"
        G_true_combined.add_node(
            record_node_id,
            label=rec_label,
            title=rec_tooltip,
            node_type='record',
            data_source=rec_source,
            type=rec_type
        )
        records_added += 1
        
        G_true_combined.add_edge(
            record_node_id,
            entity_node_id,
            edge_type='resolution'
        )

print(f"Added {entities_added} entity nodes")
print(f"Added {records_added} record nodes")
print(f"Added {G_true_combined.number_of_edges()} resolution edges")

# Add relationship edges from Senzing's RELATED_ENTITIES
relationship_edges = 0
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    
    for rel in entity.get('RELATED_ENTITIES', []):
        related_id = rel.get('ENTITY_ID')
        match_key = rel.get('MATCH_KEY', 'related')
        
        if entity_id < related_id:
            anchor_node = f"entity_{entity_id}"
            target_node = f"entity_{related_id}"
            if G_true_combined.has_node(target_node):
                G_true_combined.add_edge(
                    anchor_node,
                    target_node,
                    edge_type='relationship',
                    relationship=match_key
                )
                relationship_edges += 1

print(f"Added {relationship_edges} relationship edges")
print(f"\nTrue Combined Graph Statistics:")
print(f"  Total nodes: {G_true_combined.number_of_nodes()}")
print(f"  Total edges: {G_true_combined.number_of_edges()}")
print(f"  Entity nodes: {entities_added}")
print(f"  Record nodes: {records_added}")
print(f"  Resolution edges: {G_true_combined.number_of_edges() - relationship_edges}")
print(f"  Relationship edges: {relationship_edges}")

# Close Senzing connection
grpc_channel.close()
print("\nSenzing connection closed")

Added 196 entity nodes
Added 282 record nodes
Added 282 resolution edges
Added 492 relationship edges

True Combined Graph Statistics:
  Total nodes: 478
  Total edges: 774
  Entity nodes: 196
  Record nodes: 282
  Resolution edges: 282
  Relationship edges: 492

Senzing connection closed


## Load Embedding Model

Downloads and initializes `all-MiniLM-L6-v2` — the same model used in notebook 05 to create the vector embeddings stored in LanceDB.

In [6]:
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")
print(f"  Model: all-MiniLM-L6-v2")
print(f"  Embedding dimension: 384")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
  Model: all-MiniLM-L6-v2
  Embedding dimension: 384


## Load Pre-Created Embeddings from LanceDB

Connects to the LanceDB instance and opens the `entities` table that was populated in notebook 05. No embeddings are created here — we reuse the pre-computed vectors.

In [7]:
db = lancedb.connect('/workspace/lancedb_data')
table = db.open_table('entities')

print(f"Connected to LanceDB")
print(f"Total entities available: {table.count_rows()}")

# Preview the pre-created embeddings
print("\nSample data from LanceDB:")
sample = table.to_pandas().head(5)
display_columns = ['entity_id', 'name', 'type', 'data_sources', 'num_records', 'risks']
print(sample[display_columns].to_string())

Connected to LanceDB
Total entities available: 196

Sample data from LanceDB:
   entity_id                               name          type                   data_sources  num_records               risks
0     100001                    Abassin Badshah        PERSON  OPEN-OWNERSHIP,OPEN-SANCTIONS            3        corp.disqual
1     100002                        LMAR GB LTD  ORGANIZATION                 OPEN-SANCTIONS            1                    
2     100003            WANDLE HOLDINGS LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1     sanction.linked
3     100004  POLYUS GOLD INTERNATIONAL LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1     sanction.linked
4     100005          Firuza Nazimovna Kerimova        PERSON                 OPEN-SANCTIONS            1  role.rca; sanction


## Set Up Anthropic Client and RAG Query Function

Configures the Anthropic API client and defines the RAG pipeline: vector search in LanceDB, neighbor expansion via the knowledge graph, context assembly, and LLM query.

In [8]:
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found in environment")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Anthropic client configured with Claude Sonnet")

Anthropic client configured with Claude Sonnet


In [9]:
def ask_knowledge_graph(question, top_k=10):
    """
    Simple RAG: Search -> Expand -> Format -> Ask LLM
    Uses the True Combined Graph (entities + records + relationships).
    """
    print(f"\nQuestion: {question}")
    print("="*70)
    
    # Step 1: Vector search
    question_embedding = embedding_model.encode(question).tolist()
    results = table.search(question_embedding).limit(top_k).to_list()
    
    print(f"Found {len(results)} relevant entities")
    
    # Step 2: Collect entity IDs and expand to neighbors via G_true_combined
    entity_ids = set()
    for r in results:
        entity_ids.add(r['entity_id'])
        
        entity_node = f"entity_{r['entity_id']}"
        if entity_node in G_true_combined:
            for neighbor in list(G_true_combined.neighbors(entity_node))[:8]:
                nd = G_true_combined.nodes[neighbor]
                # Follow relationship edges to other entities
                if nd.get('node_type') == 'entity':
                    entity_ids.add(nd.get('entity_id'))
    
    print(f"Expanded to {len(entity_ids)} entities (including neighbors)")
    
    # Step 3: Build context with graph structure
    context_parts = ["ENTITIES:\n"]
    
    for entity_id in list(entity_ids)[:30]:
        entity_info = table.search().where(f"entity_id = {entity_id}").limit(1).to_list()
        if not entity_info:
            continue
        
        info = entity_info[0]
        context_parts.append(f"- {info['name']} ({info['type']})")
        context_parts.append(f"  Sources: {info['data_sources']}, Records: {info['num_records']}")
        
        if info.get('risks'):
            context_parts.append(f"  Risks: {info['risks']}")
        
        # Get relationships and record info from G_true_combined
        entity_node = f"entity_{entity_id}"
        if entity_node in G_true_combined:
            rels = []
            record_sources = []
            for neighbor in G_true_combined.neighbors(entity_node):
                nd = G_true_combined.nodes[neighbor]
                edge_data = G_true_combined.get_edge_data(entity_node, neighbor)
                
                if nd.get('node_type') == 'entity' and edge_data.get('edge_type') == 'relationship':
                    rel_type = edge_data.get('relationship', 'connected to')
                    rels.append(f"{rel_type} {nd.get('label', 'Unknown')}")
                elif nd.get('node_type') == 'record':
                    record_sources.append(nd.get('data_source', 'unknown'))
            
            if record_sources:
                context_parts.append(f"  Source records: {', '.join(record_sources)}")
            if rels:
                context_parts.append(f"  Relationships: {'; '.join(rels[:5])}")
        
        context_parts.append("")
    
    context = "\n".join(context_parts)
    
    # Step 4: Ask LLM
    print("Querying LLM...")
    
    message = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=2048,
        system="Answer questions about a corporate ownership and sanctions knowledge graph.",
        messages=[
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]
    )
    
    print("\n" + "="*70)
    print("ANSWER")
    print("="*70)
    print(message.content[0].text)
    print("="*70)

In [ ]:
print("Knowledge Graph RAG - Interactive Session")
print("="*70)
print("Ask any question about the corporate ownership and sanctions data.")
print("The system will search LanceDB and query the knowledge graph.")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()
    
    if question.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break
    
    if not question:
        continue
    
    try:
        ask_knowledge_graph(question)
        print()
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

Knowledge Graph RAG - Interactive Session
Ask any question about the corporate ownership and sanctions data.
The system will search LanceDB and query the knowledge graph.
Type 'quit' to exit.



Your question:  do you see any signatures of fraud in this dataset?



Question: do you see any signatures of fraud in this dataset?
Found 10 relevant entities
Expanded to 43 entities (including neighbors)
Querying LLM...

ANSWER
Based on this corporate ownership dataset, I can identify several **red flags and patterns** that warrant further investigation, though they don't constitute proof of fraud without additional context:

## High-Risk Indicators

### 1. **Sanctions-Related Entities**
- **Said Kerimov** appears in OPEN-SANCTIONS with risks: `role.rca; sanction`
- He has control relationships with:
  - **NUGGET CAPITAL PLC** (marked as `sanction.linked`)
  - **POLYUS CAPITAL LIMITED** (75-100% shareholding/voting)
  - Multiple aircraft and business entities through TMF CORPORATE SERVICES LIMITED
- **Family connections** to Suleyman Abusaidovich KERIMOV (also sanctioned)

### 2. **Complex Corporate Structures with Potential Obfuscation**
- **TMF CORPORATE SERVICES LIMITED** appears as registered agent/address for multiple entities connected to Said Ke

Your question:  tell me about said kerimov



Question: tell me about said kerimov
Found 10 relevant entities
Expanded to 22 entities (including neighbors)
Querying LLM...

ANSWER
# Said Kerimov - Profile

Based on the knowledge graph, here's what we know about Said Kerimov:

## **Sanctions & Risk Status**
Said Kerimov appears in **OPEN-SANCTIONS** records with the following risk indicators:
- **Sanctioned individual** (role.rca - Related to sanctioned person)
- Designated as a **sanction** target

## **Family Connections**
Said Kerimov is part of the **Kerimov family** and has documented relationships with:

1. **Suleyman Abusaidovich KERIMOV** (likely his father)
   - High-profile oligarch and politically exposed person (PEP)
   - Multiple sanctions
   - Designated as "role.oligarch" and "role.pep"

2. **Three female family members** (likely mother and sisters):
   - **Firuza Nazimovna Kerimova** (sanctioned)
   - **Amina Suleymanovna Kerimova** (sanctioned)
   - **Gulnara Suleimanova KERIMOVA** (sanctioned)

All family members